# Phase 3: The T-Learner AI Engine (XGBoost Prototyping)

Before we write professional software engineering code, we always build and train the AI in a notebook sandbox first to prove the math works!

In this notebook, we will train our **Two-Brain T-Learner Model** to hunt for 'Persuadable' customers.

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

print("Loading the clean, mathematically perfect data from the vault...")
X_train = pd.read_csv(r"../data/X_train.csv")
y_train = pd.read_csv(r"../data/y_train.csv")
X_test = pd.read_csv(r"../data/X_test.csv")
y_test = pd.read_csv(r"../data/y_test.csv")
print("Data loaded successfully!")

--- 
## Step 1: Splitting the Data into Two Timelines
We must split our `X_train` into two completely different worlds:
1. **The Treatment World:** People who saw the ad.
2. **The Control World:** People who were hidden from the ad.

In [ ]:
# Filter the training input data (X) based on who saw the ad
mask_treatment = (X_train['treatment'] == 1)
mask_control   = (X_train['treatment'] == 0)

# Create the Treatment World
X_train_treat = X_train[mask_treatment].drop(columns=['treatment'])
y_train_treat = y_train[mask_treatment]

# Create the Control World
X_train_ctrl = X_train[mask_control].drop(columns=['treatment'])
y_train_ctrl = y_train[mask_control]

print(f"Brain 1 will study {X_train_treat.shape[0]} people who saw ads.")
print(f"Brain 2 will study {X_train_ctrl.shape[0]} people who didn't.")

--- 
## Step 2: The Birth of the AI
We load two completely blank XGBoost algorithms. We will configure them to use the GPU (if available) for massive speed.

In [ ]:
# Set up the AI Brains
model_treatment = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5,
    device="cuda"  # Try to use Nvidia GPU (remove if you don't have one)
)

model_control = XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5,
    device="cuda"
)
print("AI Brains are born and ready to study.")

--- 
## Step 3: School (Training the Models)
This is where the actual Machine Learning happens. The `.fit()` function forces the AI to study the clues (`X`) and learn the secret answer key (`y`).

In [ ]:
print("Training Brain 1 (The Treatment Model)...")
model_treatment.fit(X_train_treat, y_train)

print("Training Brain 2 (The Control Model)...")
model_control.fit(X_train_ctrl, y_train_ctrl)

print("\n✅ SUCCESS! BOTH AI MODELS ARE FULLY TRAINED!")

--- 
## Step 4: Predicting Uplift (The Final Goal)
Now, we take our hidden 20% Vault data (`X_test`). We run those users through *both* Brains at the same time to see who is a Persuadable!

In [ ]:
# Hide the treatment column from the test data (we just want the features)
X_test_features = X_test.drop(columns=['treatment'])

# Ask Brain 1: What is the exact probability this person buys IF we force an ad on them?
prob_if_treated = model_treatment.predict_proba(X_test_features)[:, 1]

# Ask Brain 2: What is the exact probability this person buys IF we leave them completely alone?
prob_if_control = model_control.predict_proba(X_test_features)[:, 1]

# The Magic Formula: The Uplift Score
uplift_scores = prob_if_treated - prob_if_control

print("Uplift calculation completed!")
print("\n--- SNEAK PEEK AT OUR FIRST 5 CUSTOMERS ---")
for i in range(5):
    print(f"Customer {i}: Prob_Ad=({prob_if_treated[i]*100:.2f}%)  Prob_NoAd=({prob_if_control[i]*100:.2f}%)  -->  Absolute Uplift: {(uplift_scores[i]*100):.2f}%")